In [ ]:
# Import the libraries

import os 
import kagglehub 
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "snap/amazon-fine-food-reviews",
    "Reviews.csv",
)

df = df[["Text"]].iloc[:50_000].dropna()
df.head()

In [ ]:
# Tokenize

MAX_VOCAB_SIZE = 10_000

tokenizer = Tokenizer(
    num_words = MAX_VOCAB_SIZE,
    oov_token = "<OOV>"
)

tokenizer.fit_on_texts(df['Text'])

total_words = min(
    len(tokenizer.word_index) + 1,
    MAX_VOCAB_SIZE
)

print("Vocabulary size:", total_words)

In [ ]:
# Convert reviews to integer sequences

sequences = tokenizer.texts_to_sequences(
    df['Text']
)

In [ ]:
# Limit the sequence length

MAX_LEN = 30

sequences = [
    seq[-MAX_LEN:] for seq in sequences
]

In [ ]:
# tf.data generator
# This avoids keeping all generated training examples in RAM
SEQ_LEN = 15


def sequence_generator():

    for sequence in sequences:

        if len(sequence) < 2:
            continue

        for i in range(1, len(sequence)):

            start = max(0, i - SEQ_LEN)

            input_sequence = sequence[start:i]
            target = sequence[i]

            input_sequence = pad_sequences(
                [input_sequence],
                maxlen=SEQ_LEN,
                padding="pre"
            )[0]

            yield input_sequence, target

dataset = tf.data.Dataset.from_generator(
    sequence_generator,
    output_signature=(
        tf.TensorSpec(
            shape=(SEQ_LEN,),
            dtype=tf.int32
        ),
        tf.TensorSpec(
            shape=(),
            dtype=tf.int32
        )
    )
)

BATCH_SIZE = 128

dataset = dataset.batch(BATCH_SIZE)

dataset = dataset.prefetch(
    tf.data.AUTOTUNE
)

In [ ]:
# LSTM Model
model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=64
    ),

    LSTM(64),

    Dense(
        total_words,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Training

num_examples = sum(
    max(0, len(seq) - 1)
    for seq in sequences
)

steps_per_epoch = num_examples // BATCH_SIZE

print("Training examples:", num_examples)
print("Steps per epoch:", steps_per_epoch)

In [ ]:
history = model.fit(
    dataset,
    epochs=10,
    steps_per_epoch=steps_per_epoch
)

In [ ]:
# Prediction Function

def predict_next_word(seed_text):

    token_list = tokenizer.texts_to_sequences(
        [seed_text]
    )[0]

    token_list = token_list[-SEQ_LEN:]

    token_list = pad_sequences(
        [token_list],
        maxlen=SEQ_LEN,
        padding="pre"
    )

    predictions = model.predict(
        token_list,
        verbose=0
    )[0]

    predicted_id = np.argmax(predictions)

    for word, index in tokenizer.word_index.items():

        if index == predicted_id:
            return word

    return "<unknown>"

In [ ]:
print(
    predict_next_word(
        "this product is"
    )
)